In [ ]:
# Crawl outputs of open-gira TC/grid analysis, specifically:
# power/by_storm_set/{STORM_SET}/disruption/pop_affected_RP/{RETURN_PERIOD}.gpq files for the STORM future cases
# plot histograms (per country data points) of population disrupted for each GCM, normalised by GCM-mean
# plot scatter of global population disrupted for each GCM, normalised by GCM-mean

In [ ]:
from collections import defaultdict
import os

import geopandas as gpd
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# input paths
country_thresholds_path = "country_thresholds.csv"
wb_income_classification_path = "wb_income_classification.csv"
results_dir = "/data/open-gira/results"

# output paths
plot_dir = "plots"

# return periods to plot
return_periods = [10, 20, 50, 100, 200, 500, 1000]

# mapping from STORM_SET to plotted name
atmospheres = {
    "STORM-CNRM-CM6-1-HR": "STORM 2050 RCP8.5 CNRM",
    "STORM-CMCC-CM2-VHR4": "STORM 2050 RCP8.5 CMCC",
    "STORM-EC-Earth3P-HR": "STORM 2050 RCP8.5 EC-Earth",
    "STORM-HadGEM3-GC31-HM": "STORM 2050 RCP8.5 HadGEM3"
}

# Equilibrium Climate Sensitivity (ECS) for each of the GCMs
ecs = {
    "STORM-CNRM-CM6-1-HR": 4.28,
    "STORM-CMCC-CM2-VHR4": 3.52,  # this is for CMCC-CM2-SR5 as I can't find VHR4
    "STORM-EC-Earth3P-HR": 4.31,
    "STORM-HadGEM3-GC31-HM": 5.42,  # this is for HadGEM3-GC31-MM
}
ecs = pd.Series(index=ecs.keys(), data=ecs.values())

In [ ]:
def plot_histogram_normalised_disruption_by_gcm(
    return_period: int,
    norm_df: pd.DataFrame,
    ecs: pd.Series
) -> None:
    
    f, axes = plt.subplots(4, 1, sharex=True)
    bins = np.linspace(0, 2, 20)
    sorted_by_ecs = norm_df.loc[ecs.sort_values().index, :]
    ymax = 0
    
    # plot histograms and calculate ymax
    for i, row in enumerate(sorted_by_ecs.itertuples()):
        gcm_data = norm_df.loc[row.Index, :]
        axes[i].hist(gcm_data, bins=bins, alpha=0.5)
        _, ax_ymax = axes[i].get_ylim()
        ymax = max([ax_ymax, ymax])
    
    # plot everything else (including things that depend on ymax)
    for i, row in enumerate(sorted_by_ecs.itertuples()):
        gcm_data = norm_df.loc[row.Index, :]
        axes[i].axvline(gcm_data.mean(), ls="--", c="r")
        axes[i].set_ylim(0, 1.05 * ymax)
        axes[i].text(
            gcm_data.mean() * 1.02,
            0.05 * ymax,
            r"$\mu=$" + f"{gcm_data.mean():.3f}",
            horizontalalignment='left',
            verticalalignment='bottom',
            rotation=90,
            fontsize=7,
        )
        axes[i].text(
            0.05,
            0.9 * ymax,
            f"{'-'.join(row.Index.split('-')[1:])}, ECS: {ecs[row.Index]} [K]",
            horizontalalignment='left',
            verticalalignment='top',
            fontsize=8
        )
        axes[i].set_yticklabels([])
        axes[i].grid(alpha=0.5)
        axes[i].set_xlim(0, 2)
    f.supxlabel('Normalized population affected')
    f.supylabel('Frequency', x=0.07)
    f.suptitle(f"Disruption by GCM, 1-in-{return_period} year event", y=0.93)
    f.savefig(f"plots/norm_disrupt_by_gcm_{return_period}.png")
    plt.close(f)

In [ ]:
def plot_scatter_normalised_disruption_by_gcm(
    means_by_return_period: dict[int, pd.DataFrame],
    ecs: pd.DataFrame,
    plot_dir: str,
) -> None:

    df = pd.DataFrame(means_by_return_period)
    
    f, ax = plt.subplots(figsize=(4.5,3))
    cmap = matplotlib.colormaps["magma_r"]
    markers = ["*", "x", "+", "^"]
    for i, gcm in enumerate(ecs.sort_values(ascending=False).index): # reverse so highest ECS is first
        short_gcm = '-'.join(gcm.split('-')[1:])
        ax.scatter(
            df.columns.values,
            df.loc[gcm, :],
            marker=markers[i],
            label=f"{ecs[gcm]}, {short_gcm}",
            color=cmap(i/len(ecs) + 0.1)
        )
        
    legend = ax.legend(
        title="",
        fontsize=6,
        ncols=2,
        loc="upper center",
    )
    legend.set_title("Equilibrium Climate Sensitivity [K], General Circulation Model", prop={'size': 6})
    legend._legend_box.sep = 4
    
    ax.axhline(1, ls="--", c="k", alpha=0.2)
    ax.set_xscale("log")
    ymin, ymax = ax.get_ylim()
    ax.set_ylim(0.97 * ymin, ymax * 1.07)
    ax.grid(which="both", alpha=0.3)
    ax.set_xlabel("Return period [years]", fontsize=9, labelpad=8)
    ax.set_ylabel("Normalized population affected", fontsize=9, labelpad=10)
    ax.tick_params(axis='y', which='major', labelsize=9)
    plt.subplots_adjust(left=0.2, right=0.92, bottom=0.21, top=0.93)
    
    f.savefig(os.path.join(plot_dir, f"norm_disrupt_gcm_scatter.png"), dpi=254)

In [ ]:
def lookup(df: pd.DataFrame, rows, cols) -> np.ndarray:
    """
    Helper function to lookup values based on list of `rows` and `cols` labels.
    Does this really not exist in pandas already?!

    Args:
        df: Table to lookup from
        rows: Iterable of row indicies to select
        cols: Iterable of column indicies to select (same length as `rows`)

    Returns:
        Elements of `df` that are indexed by `rows` and `cols`.
    """
    return df.to_numpy()[df.index.get_indexer(rows), df.columns.get_indexer(cols)]

In [ ]:
# assemble optimum country to threshold mapping
# calibrated thresholds, ms-1
per_country_calibrated_thresholds = pd.read_csv(country_thresholds_path).rename(columns={"iso_a3": "ISO_A3"}).set_index("ISO_A3")
# thresholds based on income, ms-1
income_threshold_map = {"L": "30.0", "LM": "30.0", "UM": "30.0", "H": "32.5"}
thresholds = pd.read_csv(wb_income_classification_path, comment="#").set_index("ISO_A3")
thresholds["threshold_ms-1"] = thresholds.INCOME_LEVEL.map(income_threshold_map)
thresholds.loc[per_country_calibrated_thresholds.index, "threshold_ms-1"] = per_country_calibrated_thresholds["threshold_ms-1"]
thresholds = thresholds[~thresholds["threshold_ms-1"].isna()]

In [ ]:
# plot some distributions of people affected per GCM, normalised by the GCM mean
for return_period in return_periods:
    data = {}
    for atmosphere in atmospheres.keys():
        path = os.path.join(results_dir, f"power/by_storm_set/{atmosphere}/disruption/pop_affected_RP/{return_period}.gpq")
        pop_at_risk = gpd.read_parquet(path)
        # country row index
        series = pd.Series(
            index=thresholds.index,
            data=lookup(pop_at_risk, thresholds.index, thresholds["threshold_ms-1"].values.astype(str))
        )
        # filter out zero values (avoids divide by zero when normalising later)
        series = series[series > 0]
        data[atmosphere] = series

    # GCM row index, country column index
    df = pd.DataFrame(data).T

    # GCM row index, country column index
    norm_df = df / df.mean(axis=0)
    
    plot_histogram_normalised_disruption_by_gcm(return_period, norm_df, ecs)

In [ ]:
# again, take the calibrated population at risk figures
# but now sum across the countries, and normalise to the mean of the GCMs 
# find these figures for each return period and GCM
means_by_return_period = {}
for return_period in return_periods:
    data = {}
    for atmosphere in atmospheres.keys():
        path = os.path.join(results_dir, f"power/by_storm_set/{atmosphere}/disruption/pop_affected_RP/{return_period}.gpq")
        pop_at_risk = gpd.read_parquet(path)
        series = pd.Series(
            index=thresholds.index,
            data=lookup(pop_at_risk, thresholds.index, thresholds["threshold_ms-1"].values.astype(str))
        )
        data[atmosphere] = series
        
    df = pd.DataFrame(data).sum()
    norm_df = (df / df.mean()).to_frame()

    sorted_by_ecs = norm_df.loc[ecs.sort_values().index]
    for i, row in enumerate(sorted_by_ecs.itertuples()):
        means_by_return_period[return_period] = norm_df.mean(axis=1)

In [ ]:
# plot a scatter of people at risk as a function of return period
plot_scatter_normalised_disruption_by_gcm(means_by_return_period, ecs, plot_dir)